# Gates de qualidade de dados
Falha o job se houver: divergência entre o manifesto do extrator e a Bronze, nulos ou duplicidades na chave da Silver, registros órfãos na fato ou tabelas da Gold sem COMMENT.

In [ ]:
for nome, padrao in [("catalogo_bronze", "dev_bronze"), ("catalogo_silver", "dev_silver"),
                     ("catalogo_gold", "dev_gold"), ("landing_path", "/Volumes/landing/protheus/arquivos"),
                     ("src_path", ""), ("dias_reconciliacao", "7")]:
    dbutils.widgets.text(nome, padrao)

import sys

src_path = dbutils.widgets.get("src_path")
if src_path and src_path not in sys.path:
    sys.path.append(src_path)

In [ ]:
from datetime import UTC, datetime, timedelta
from functools import reduce

from pyspark.sql import functions as F

from pdc_lib.qualidade import contar_duplicidades, contar_nulos, contar_orfaos, reconciliar
from pdc_lib.util import nome_tabela, validar_identificador

catalogo_bronze = validar_identificador(dbutils.widgets.get("catalogo_bronze"))
catalogo_silver = validar_identificador(dbutils.widgets.get("catalogo_silver"))
catalogo_gold = validar_identificador(dbutils.widgets.get("catalogo_gold"))
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
dias = int(dbutils.widgets.get("dias_reconciliacao"))

TABELAS = ["sc5", "sc6", "sa1", "sa3", "sb1"]
falhas: list[str] = []
fato_nome = nome_tabela(catalogo_gold, "comercial", "fato_pedido_venda")
if not spark.catalog.tableExists(fato_nome):
    dbutils.notebook.exit("Gold ainda não disponível; gates não aplicáveis.")

In [ ]:
# 1. Chave da Silver: sem nulos e sem duplicidades (empresa + R_E_C_N_O_)
for tabela in TABELAS:
    df = spark.table(nome_tabela(catalogo_silver, "protheus", tabela))
    chave = ["cod_empresa", "num_recno"]
    if (nulos := contar_nulos(df, chave)) > 0:
        falhas.append(f"silver.{tabela}: {nulos} registros com chave nula")
    if (duplicados := contar_duplicidades(df, chave)) > 0:
        falhas.append(f"silver.{tabela}: {duplicados} chaves duplicadas")

In [ ]:
# 2. Integridade da fato com as dimensões
fato = spark.table(fato_nome)
relacoes = {
    "dim_cliente": ["cod_empresa", "cod_cliente", "cod_loja"],
    "dim_vendedor": ["cod_empresa", "cod_vendedor"],
    "dim_produto": ["cod_empresa", "cod_produto"],
    "dim_unidade": ["cod_empresa", "cod_filial"],
}
for dimensao, chaves in relacoes.items():
    orfaos = contar_orfaos(fato, spark.table(nome_tabela(catalogo_gold, "comercial", dimensao)), chaves)
    if orfaos > 0:
        falhas.append(f"fato_pedido_venda: {orfaos} combinações sem correspondência em {dimensao}")

In [ ]:
# 3. Reconciliação: registros declarados no manifesto do extrator x registros na Bronze (últimos N dias)
corte = (datetime.now(UTC) - timedelta(days=dias)).strftime("%Y%m%dT%H%M%SZ")
try:
    manifestos = spark.read.option("multiLine", "true").json(f"{landing_path}/_manifest/*.json")
except Exception:
    manifestos = None

if manifestos is not None and "itens" in manifestos.columns:
    itens = (
        manifestos.filter(F.col("lote") >= corte)
        .select("lote", F.explode("itens").alias("item"))
        .select("lote", "item.tabela", "item.empresa", F.col("item.registros").cast("long").alias("registros"))
    )
    contagens = []
    for tabela in TABELAS:
        bronze_nome = nome_tabela(catalogo_bronze, "protheus", tabela)
        if spark.catalog.tableExists(bronze_nome) and "_lote" in spark.table(bronze_nome).columns:
            contagens.append(
                spark.table(bronze_nome).filter(F.col("_lote") >= corte)
                .groupBy(F.col("_lote").alias("lote"), F.col("_empresa").alias("empresa"))
                .agg(F.count("*").alias("qtd_bronze"))
                .withColumn("tabela", F.lit(tabela))
            )
    if contagens:
        divergencias = reconciliar(itens, reduce(lambda a, b: a.unionByName(b), contagens)).collect()
        for d in divergencias:
            falhas.append(f"reconciliação {d.tabela}/{d.empresa} lote {d.lote}: "
                          f"manifesto={d.registros}, bronze={d.qtd_bronze}")
else:
    print("Nenhum manifesto encontrado; reconciliação não aplicada.")

In [ ]:
# 4. Documentação: todas as tabelas da Gold com COMMENT
sem_comentario = spark.sql(f"""
    SELECT table_name FROM {catalogo_gold}.information_schema.tables
    WHERE table_schema = 'comercial' AND (comment IS NULL OR trim(comment) = '')
""").collect()
for linha in sem_comentario:
    falhas.append(f"{catalogo_gold}.comercial.{linha.table_name}: tabela sem COMMENT")

In [ ]:
if falhas:
    raise AssertionError("Gates de qualidade reprovados:\n- " + "\n- ".join(falhas))
print("Todos os gates de qualidade de dados foram aprovados.")